# BangHallu — Pretrained encoders on GPU (M4 + M5)

This is the first rung of the ladder that has seen **text from outside this corpus**.
Everything before it — counting words, our own Skip-gram, the from-scratch RNNs and
Transformer — learned only from the 6,134 training records.

The measurement this notebook exists to make:

> `transformer_scratch` and BanglaBERT are the **same architecture family**.
> One has pretraining, one does not. **The gap between them is what pretraining is worth.**

| Encoder | Hugging Face ID | Why it is here |
|---|---|---|
| `banglabert` | `csebuetnlp/banglabert` | Bengali-only, the PRD's main Phase 1 model |
| `muril` | `google/muril-base-cased` | 17 Indian languages — **required by M5** |
| `xlmr_base` / `xlmr_large` | `xlm-roberta-base` / `-large` | the reference notebooks' pick |
| `mbert` | `bert-base-multilingual-cased` | the original multilingual BERT |

**Runtime → Change runtime type → T4 GPU.** `xlmr_large` needs more VRAM; drop the batch size
to 8 if it runs out.

**The test split is never opened here.** Train on `train`, score on `dev`.

## What this takes from the two reference notebooks, and what it deliberately does not

**Taken:**
- **Two-stage fine-tuning** (`--two-stage`): train the head with the encoder frozen, then
  unfreeze only the top few layers at a lower rate. Both notebooks found full fine-tuning
  collapsed to near-chance on their ~250 rows.
- **Explicit role markers** in the input (`প্রসঙ্গ:` / `প্রশ্ন:` / `উত্তর:`), giving the model
  a fixed place to find each part.

**Not taken, and why:**
- **Back-translation augmentation.** Their single biggest lever — but it manufactures training
  rows, and this project's rule is that the corpus is final. Their datasets were ~250 rows;
  yours is 6,134, so the problem it solves barely applies here.
- **The Qwen-7B LLM judge as an ensemble feature.** Your PRD defines an LLM as M8, a
  *reference point* reported beside your models. Blending it in would mean your final system
  contains a 7B model, which changes what the project is claiming.

**One thing they both miss:** neither runs `csebuetnlp/normalizer` before BanglaBERT.
BanglaBERT was pretrained on normalised text, so skipping it is a silent accuracy loss.
This code applies it — and **refuses to run** BanglaBERT without it rather than quietly
training a worse model.

## 1. Install what Colab does not already have

In [ ]:
# transformers is usually present; the normaliser never is.
!pip install -q "transformers>=4.40" sentencepiece accelerate
!pip install -q git+https://github.com/csebuetnlp/normalizer
print("done — if the normaliser failed, BanglaBERT will refuse to run (by design)")

## 2. Mount Drive and find the project

In [ ]:
import sys, pathlib, importlib

REPO = "/content/drive/MyDrive/Bhibranti"      # <- change if you put it somewhere else

# colab_setup.py lives ON Drive, so Drive must be mounted before it can be imported.
# (prepare() also mounts, but that runs after the import below, which is too late.)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass                                        # not on Colab: the repo is already local

setup_file = pathlib.Path(REPO) / "src" / "colab_setup.py"
if not setup_file.exists():
    parent = pathlib.Path(REPO).parent
    raise FileNotFoundError(
        f"{setup_file} not found. Upload the whole project folder (src/, data/, ...) to Drive "
        f"and set REPO above to its path. Under {parent} right now: "
        f"{sorted(p.name for p in parent.glob('*'))[:20]}")

# If this cell ran before Drive was mounted, Python cached "src/ does not exist" and would
# keep skipping it even now. Clear that, or the import fails although the file is there.
sys.path.insert(0, str(pathlib.Path(REPO) / "src"))
importlib.invalidate_caches()
sys.path_importer_cache.pop(str(pathlib.Path(REPO) / "src"), None)
sys.modules.pop("colab_setup", None)

from colab_setup import prepare

paths = prepare(repo=REPO)

## 3. Check the guards before downloading a single model

This costs nothing and needs no GPU. It shows exactly what text each encoder will be given,
confirms the answer always survives into the second segment (the passage is what gets cut,
never the answer), and proves BanglaBERT refuses to run without its normaliser.

In [ ]:
import subprocess

done = subprocess.run([sys.executable, "src/train_transformer.py", "--check"],
                      cwd=paths.root, capture_output=True, text=True)
print(done.stdout)
assert done.returncode == 0, "a guard failed — fix it before training"

## 4. Fine-tune BanglaBERT

The PRD recipe (guide §7.3): AdamW, learning rate 2e-5, batch 32, up to 5 epochs, 10% warm-up,
early stopping on dev loss with patience 2, keeping the best epoch.

Watch the same thing as in notebook 1: **dev loss turning back upward while train loss keeps
falling.** If the run collapses to answering one class for everything, the code says so
outright — and then `--two-stage` in section 6 is the fix.

In [ ]:
import train_transformer as tt
from splits import load_split

train, dev = load_split("train"), load_split("dev")

result, target, notes, fitted, scores = tt.run_one(
    "banglabert", seed=42, two_stage=False, train=train, dev=dev)

## 5. The number that matters

`has-context + hard` is the PRD G4 target. The full report above also breaks the score down by
condition, difficulty, subject and error type, and recomputes the no-learning rules on exactly
these records — because a model that cannot beat a two-line string matcher has shown nothing.

In [ ]:
print(f"dev macro-F1        : {result.macro_f1:.3f}")
print(f"has-context + hard  : {target:.3f}    <- the PRD G4 target (>= 0.80)")
print(f"best epoch          : {fitted.best_epoch} of {fitted.epochs_run}")
print(f"recipe              : {notes['recipe']}")
if "WARNING" in notes:
    print("WARNING:", notes["WARNING"])

## 6. If it collapsed — the two-stage recipe

Only worth running if section 4 printed a collapse warning, or scored near 0.33.

Stage 1 trains the classifier head with the encoder completely frozen, so a large learning
rate is safe: nothing pretrained can move. Stage 2 unfreezes only the top 4 layers at a much
lower rate. The lower layers keep the general Bengali knowledge that pretraining paid for.

In [ ]:
result2, target2, notes2, fitted2, _ = tt.run_one(
    "banglabert", seed=42, two_stage=True, train=train, dev=dev)

print(f"one-stage : {result.macro_f1:.3f} overall / {target:.3f} hard")
print(f"two-stage : {result2.macro_f1:.3f} overall / {target2:.3f} hard")

## 7. The comparison that justifies this whole step

`transformer_scratch` (notebook 1) and BanglaBERT share an architecture family. The only real
difference is pretraining. Put the two numbers side by side.

McNemar is used rather than eyeballing the gap, because the guide requires it before any claim
that one model beats another.

In [ ]:
import evaluate
import csv

# the from-scratch Transformer's score, read from the log rather than retyped
rows = [r for r in csv.DictReader(open(paths.root / "results" / "experiment_log.csv", encoding="utf-8"))
        if r["model"] == "M13_transformer_scratch" and r["dev_macro_f1"]]
if rows:
    scratch = sum(float(r["dev_macro_f1"]) for r in rows) / len(rows)
    print(f"Transformer, no pretraining (M13) : {scratch:.3f}   ({len(rows)} runs)")
print(f"BanglaBERT, pretrained      (M4) : {result.macro_f1:.3f}")

hard = [i for i, r in enumerate(dev) if r["context"] and r["difficulty"] == "hard"]
truth = [dev[i]["label"] for i in hard]
model_pred = [int(scores[i] >= 0.5) for i in hard]
fuzzy_all = evaluate.baseline_predictions(dev, "fuzzy")
fuzzy = [fuzzy_all[i] for i in hard]

wins_model, wins_rule, p = evaluate.mcnemar(truth, model_pred, fuzzy)
print(f"\non {len(hard)} hard, has-context dev records:")
print(f"  BanglaBERT  {evaluate.macro_f1(truth, model_pred):.3f}")
print(f"  fuzzy rule  {evaluate.macro_f1(truth, fuzzy):.3f}")
print(f"  disagree on {wins_model + wins_rule}: model right {wins_model}, rule right {wins_rule}")
print(f"  McNemar p = {p:.3f}  ->", "too close to call" if p > 0.05 else "a real difference")

## 8. The full comparison, and recording it

Every encoder, three seeds each. This is the expensive cell — budget an hour or more on a T4.
Start with two or three encoders rather than all seven.

A warning from the guide worth keeping in mind: in a published benchmark on a related Bangla
task, **seven very different models scored within 2.8 points of each other**. Do not expect the
choice of architecture alone to transform the result — input format and further pretraining
move the needle more. Report ties honestly.

In [ ]:
# Trim this list to what you have GPU time for. xlmr_large wants a smaller batch.
!cd "{REPO}" && python src/train_transformer.py --model banglabert --seeds --log --note "Colab GPU"

In [ ]:
# Or, in-process, several encoders in a row:
for name in ["banglabert", "muril", "xlmr_base"]:
    print(f"\n===== {name} =====", flush=True)
    r, t, n, f, _ = tt.run_one(name, seed=42, two_stage=False, train=train, dev=dev, quiet=True)
    print(f"{name}: dev {r.macro_f1:.3f} | hard {t:.3f} | {n.get('WARNING', 'ok')}")
    print("logged as", tt.log_result(name, 42, r, t, n, f, "Colab GPU"))

## 9. Reading the result honestly

- **If BanglaBERT clearly beats `transformer_scratch`**, that difference is what pretraining
  bought — the single most useful number in this project so far.
- **If it does not beat the fuzzy string rule on the hard subset**, say so. That is a real
  finding about how hard this task is, not a failure to hide.
- **If the encoders all land within a couple of points of each other**, that matches the
  published benchmark quoted above. Report it as a tie, with McNemar, rather than crowning
  whichever one happened to come first.

Dev decided when training stopped and which epoch was kept, so these dev scores are mildly
optimistic. `test` is still untouched and is scored exactly once, at the end.

**Next steps on the ladder:** M6 further pretraining, M7 ensembling, M8 the LLM reference
point — kept separate from the model, by design.